In [27]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AdamW, get_linear_schedule_with_warmup, AutoModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd

import torch.nn as nn
import torch.nn.functional as F

In [28]:
# Load the IndoBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("C:/Users/bdrex/Documents/Huggingface Model/indobert-base-uncased")

In [29]:
# Hyperparameters
max_len = 128
batch_size = 16
epochs = 5
learning_rate = 2e-5
num_classes = 13
lstm_hidden_size = 128  # Number of LSTM hidden units
embed_dim = 768

In [30]:
class AttentionLayer(nn.Module):
    def __init__(self, hidden_size):
        super(AttentionLayer, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, lstm_output):
        attention_weights = F.softmax(self.attention(lstm_output), dim=1)
        context_vector = torch.sum(attention_weights * lstm_output, dim=1)
        return context_vector

class TextClassificationModel(nn.Module):
    def __init__(self, n_classes, lstm_hidden_size=128, dropout_rate=0.3):
        super(TextClassificationModel, self).__init__()
        self.bert = AutoModel.from_pretrained("C:/Users/bdrex/Documents/Huggingface Model/indobert-base-uncased")
        self.lstm = nn.LSTM(self.bert.config.hidden_size, lstm_hidden_size, batch_first=True, bidirectional=True)
        self.attention = AttentionLayer(lstm_hidden_size * 2)
        
        self.fc1 = nn.Linear(lstm_hidden_size * 2 + self.bert.config.hidden_size, 256)
        self.fc2 = nn.Linear(256, 64)
        self.out = nn.Linear(64, n_classes)
        
        self.layer_norm1 = nn.LayerNorm(256)
        self.layer_norm2 = nn.LayerNorm(64)
        
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):
        # BERT layer
        bert_output, pooled_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False
        )
        
        # LSTM layer
        lstm_output, _ = self.lstm(bert_output)
        
        # Attention layer
        attn_output = self.attention(lstm_output)
        
        # Concatenate BERT pooled output and LSTM attention output
        combined = torch.cat((pooled_output, attn_output), dim=1)
        
        # Fully connected layers with residual connections and layer normalization
        fc1_output = self.fc1(combined)
        fc1_output = self.layer_norm1(fc1_output + self.dropout(self.relu(fc1_output)))
        
        fc2_output = self.fc2(fc1_output)
        fc2_output = self.layer_norm2(fc2_output + self.dropout(self.relu(fc2_output)))
        
        # Output layer
        output = self.out(fc2_output)
        return output

In [31]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [32]:
df = pd.read_csv('target_news.csv', index_col=0)

In [33]:
label_2_idx = {y:x for x,y in enumerate(sorted(df.new_category.unique()))}
idx_2_label = {x:y for x,y in enumerate(sorted(df.new_category.unique()))}

In [34]:
df['new_category'] = df['new_category'].map(label_2_idx)

In [35]:
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_test = train_test_split(df_train, test_size=0.2, random_state=42)

df_train = df_train.reset_index()
df_val = df_val.reset_index()
df_test = df_test.reset_index()

X_train, X_val, X_test = df_train['headline'], df_val['headline'], df_test['headline']
y_train, y_val, y_test = df_train['new_category'], df_val['new_category'], df_test['new_category']

In [36]:
train_dataset = TextClassificationDataset(X_train, y_train, tokenizer, max_len)
val_dataset = TextClassificationDataset(X_val, y_val, tokenizer, max_len)
test_dataset = TextClassificationDataset(X_test, y_test, tokenizer, max_len)

train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_data_loader = DataLoader(val_dataset, batch_size=batch_size)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size)

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextClassificationModel(num_classes)
model = model.to(device)

In [38]:
optimizer = AdamW(model.parameters(), lr=learning_rate, correct_bias=False)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
total_steps = len(train_data_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

c:\Users\bdrex\anaconda3\lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [39]:
# Training loop
for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_predictions = 0
    for batch in tqdm(train_data_loader, desc=f"Training Epoch {epoch + 1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        train_loss += loss.item()
        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == labels)
    train_accuracy = correct_predictions.double() / len(train_dataset)
    print(f"Validation Loss: {loss/len(train_data_loader):.4f}")
    print(f"Validation Accuracy: {train_accuracy:.4f}")

    # Validation
    model.eval()
    val_loss = 0
    correct_predictions = 0
    with torch.no_grad():
        for batch in tqdm(val_data_loader, desc="Validating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = nn.CrossEntropyLoss()(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, dim=1)
            correct_predictions += torch.sum(preds == labels)

    val_accuracy = correct_predictions.double() / len(val_dataset)
    print(f"Validation Loss: {val_loss/len(val_data_loader):.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

Validating: 100%|██████████| 124/124 [00:11<00:00, 10.50it/s]


Validation Loss: 2.2297
Validation Accuracy: 0.2604


Validating: 100%|██████████| 124/124 [00:11<00:00, 10.41it/s]


Validation Loss: 2.2322
Validation Accuracy: 0.2604


Training Epoch 3:  30%|███       | 120/394 [00:38<01:27,  3.12it/s]


KeyboardInterrupt: 

In [40]:
print(labels)

tensor([ 1,  8, 11,  8, 10, 10, 10,  6, 12,  8,  3, 10,  7,  8, 10,  8],
       device='cuda:0')


In [ ]:
# Validation
model.eval()
test_loss = 0
correct_predictions = 0
with torch.no_grad():
    for batch in tqdm(test_data_loader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        test_loss += loss.item()
        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == labels)

test_accuracy = correct_predictions.double() / len(test_dataset)
print(f"Test Loss: {test_loss/len(test_data_loader):.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Function to predict category for new text
def predict_category(text):
    model.eval()
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        print(outputs)
        _, preds = torch.max(outputs, dim=1)
    
    return preds.item()

In [ ]:
test_text = X_test.iloc[997]
print(test_text)
print(idx_2_label[predict_category(test_text)])